
# Simplified RPC Pseudo-Tracklet Notebook for Students

This notebook presents a **didactic and self-contained** version of RPC pseudo-tracklet efficiency analysis.

## Educational goal
The purpose here is **not** to reproduce the full research pipeline, but to explain the core logic in a way that students can understand and modify easily.

## Main ideas
We will:
1. generate a simple set of particle crossings,
2. simulate hits in several RPC chambers,
3. reconstruct a pseudo-tracklet from reference chambers,
4. predict where the particle should hit a target chamber,
5. decide whether the target chamber has a compatible hit,
6. compute the efficiency.

## Why this simplified model?
The full analysis used in research often depends on:
- ROOT files,
- many detector-specific branches,
- clustering logic,
- HV scan wrappers,
- plotting conventions.

For teaching, it is often better to start from a **minimal physical picture**:
- chambers are planes,
- tracks are approximately straight,
- hits have finite spatial and timing resolution,
- matching is controlled by simple tolerances.

Once this logic is clear, the same structure can later be extended to real data.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["font.size"] = 11



## 1. Geometry of the simplified detector

We model four parallel RPC chambers placed at different positions along the beam direction $z$:

- Chamber A at $z_A$
- Chamber B at $z_B$
- Chamber C at $z_C$
- Chamber D at $z_D$

A particle moves approximately along a straight line:
$$
x(z) = x_0 + m_x z, \qquad y(z) = y_0 + m_y z.
$$

We also associate a reference time $t_0$, and each chamber measures the crossing time with some time resolution.


In [ ]:

z_positions = {
    "A": 0.0,
    "B": 50.0,
    "C": 100.0,
    "D": 150.0,
}

z_positions



## 2. Simulating a simple dataset

We generate artificial events with:
- an initial point $(x_0, y_0)$,
- small slopes $(m_x, m_y)$,
- a reference time $t_0$.

Then we simulate detector measurements by adding:
- spatial noise,
- timing noise,
- optional inefficiency in one chamber.

This is enough to teach the logic of efficiency reconstruction.


In [ ]:

N_EVENTS = 4000

# true track parameters
x0 = rng.uniform(-20, 20, N_EVENTS)
y0 = rng.uniform(-60, 60, N_EVENTS)
mx = rng.normal(0.00, 0.05, N_EVENTS)   # cm per cm in z
my = rng.normal(0.00, 0.08, N_EVENTS)
t0 = rng.normal(0.0, 1.0, N_EVENTS)

truth = pd.DataFrame({
    "event": np.arange(N_EVENTS),
    "x0": x0,
    "y0": y0,
    "mx": mx,
    "my": my,
    "t0": t0,
})

truth.head()



## 3. Detector response model

Each chamber measures:
- $x$,
- $y$,
- $t$.

We use:
- Gaussian spatial resolution,
- Gaussian timing resolution,
- chamber efficiency.

If a chamber fails to detect the event, the corresponding hit is marked as missing.


In [ ]:

sigma_x = 0.8   # cm
sigma_y = 1.2   # cm
sigma_t = 0.6   # ns

efficiencies = {
    "A": 0.98,
    "B": 0.97,
    "C": 0.93,   # treat C as the chamber under study
    "D": 0.96,
}

def true_position_at_z(df, z):
    x_true = df["x0"].to_numpy() + df["mx"].to_numpy() * z
    y_true = df["y0"].to_numpy() + df["my"].to_numpy() * z
    t_true = df["t0"].to_numpy() + 0.02 * z
    return x_true, y_true, t_true

def simulate_hits(df, z_positions, efficiencies, sigma_x, sigma_y, sigma_t, rng):
    records = []
    for chamber, z in z_positions.items():
        x_true, y_true, t_true = true_position_at_z(df, z)
        seen = rng.random(len(df)) < efficiencies[chamber]

        x_meas = np.where(seen, x_true + rng.normal(0, sigma_x, len(df)), np.nan)
        y_meas = np.where(seen, y_true + rng.normal(0, sigma_y, len(df)), np.nan)
        t_meas = np.where(seen, t_true + rng.normal(0, sigma_t, len(df)), np.nan)

        chamber_df = pd.DataFrame({
            "event": df["event"],
            "chamber": chamber,
            "z": z,
            "seen": seen,
            "x": x_meas,
            "y": y_meas,
            "t": t_meas,
        })
        records.append(chamber_df)
    return pd.concat(records, ignore_index=True)

hits = simulate_hits(truth, z_positions, efficiencies, sigma_x, sigma_y, sigma_t, rng)
hits.head(10)



## 4. Wide-format event table

For teaching, it is often easier to put each chamber on the same row.


In [ ]:

wide = hits.pivot(index="event", columns="chamber", values=["seen", "x", "y", "t"])
wide.columns = [f"{name}_{ch}" for name, ch in wide.columns]
wide = wide.reset_index()

data = truth.merge(wide, on="event")
data.head()



## 5. Visualizing a few true tracks

This plot shows the true trajectories in the $x$-$z$ plane for a few events.


In [ ]:

sample_events = truth.sample(8, random_state=1)

z_grid = np.linspace(0, 150, 100)
plt.figure()
for _, row in sample_events.iterrows():
    x_line = row["x0"] + row["mx"] * z_grid
    plt.plot(z_grid, x_line, alpha=0.8)
for chamber, z in z_positions.items():
    plt.axvline(z, linestyle="--", alpha=0.5)
    plt.text(z + 1, plt.ylim()[1]*0.9 if plt.ylim()[1] else 0, chamber)
plt.xlabel("z position [cm]")
plt.ylabel("x position [cm]")
plt.title("Example true trajectories in the x-z plane")
plt.show()



## 6. Baseline idea: two-chamber logic $A \rightarrow C$

A simple educational version is:
- use chamber A as reference,
- compare directly with chamber C,
- count a match if the difference stays within tolerances.

This is easy to understand, but it is weaker than a true multi-chamber pseudo-tracklet because:
- it uses less geometric information,
- it is more sensitive to fluctuations and noise.


In [ ]:

tol_x = 3.0  # cm
tol_y = 5.0  # cm
tol_t = 2.0  # ns

baseline = data.copy()

baseline["eligible_A_to_C"] = baseline["seen_A"] == True
baseline["matched_A_to_C"] = (
    baseline["eligible_A_to_C"] &
    (baseline["seen_C"] == True) &
    (np.abs(baseline["x_C"] - baseline["x_A"]) < tol_x) &
    (np.abs(baseline["y_C"] - baseline["y_A"]) < tol_y) &
    (np.abs(baseline["t_C"] - baseline["t_A"] - 0.02*(z_positions["C"] - z_positions["A"])) < tol_t)
)

N_eligible_base = int(baseline["eligible_A_to_C"].sum())
N_matched_base = int(baseline["matched_A_to_C"].sum())
eff_base = N_matched_base / N_eligible_base

print("Baseline two-chamber efficiency estimate:")
print(f"Eligible events = {N_eligible_base}")
print(f"Matched events  = {N_matched_base}")
print(f"Efficiency      = {eff_base:.4f}")



## 7. Multi-chamber pseudo-tracklet: $(A+B) \rightarrow C$

Now we use **two reference chambers**, A and B, to predict the hit in chamber C.

### Step 1
Estimate the track slopes from A and B:
$$
m_x \approx \frac{x_B - x_A}{z_B - z_A}, \qquad
m_y \approx \frac{y_B - y_A}{z_B - z_A}.
$$

### Step 2
Extrapolate to chamber C:
$$
x_C^{\mathrm{pred}} = x_A + m_x (z_C - z_A),
\qquad
y_C^{\mathrm{pred}} = y_A + m_y (z_C - z_A).
$$

### Step 3
Estimate the expected time in chamber C with a simple linear interpolation.

### Step 4
Check whether the measured hit in C lies inside the allowed compatibility window.


In [ ]:

def predict_target_from_two_refs(df, ref1, ref2, target, z_positions):
    z1 = z_positions[ref1]
    z2 = z_positions[ref2]
    zt = z_positions[target]

    mx_est = (df[f"x_{ref2}"] - df[f"x_{ref1}"]) / (z2 - z1)
    my_est = (df[f"y_{ref2}"] - df[f"y_{ref1}"]) / (z2 - z1)
    mt_est = (df[f"t_{ref2}"] - df[f"t_{ref1}"]) / (z2 - z1)

    x_pred = df[f"x_{ref1}"] + mx_est * (zt - z1)
    y_pred = df[f"y_{ref1}"] + my_est * (zt - z1)
    t_pred = df[f"t_{ref1}"] + mt_est * (zt - z1)

    return x_pred, y_pred, t_pred

study = data.copy()

study["eligible_AB_to_C"] = (
    (study["seen_A"] == True) &
    (study["seen_B"] == True)
)

x_pred, y_pred, t_pred = predict_target_from_two_refs(study, "A", "B", "C", z_positions)

study["xC_pred"] = x_pred
study["yC_pred"] = y_pred
study["tC_pred"] = t_pred

study["matched_AB_to_C"] = (
    study["eligible_AB_to_C"] &
    (study["seen_C"] == True) &
    (np.abs(study["x_C"] - study["xC_pred"]) < tol_x) &
    (np.abs(study["y_C"] - study["yC_pred"]) < tol_y) &
    (np.abs(study["t_C"] - study["tC_pred"]) < tol_t)
)

N_eligible = int(study["eligible_AB_to_C"].sum())
N_matched = int(study["matched_AB_to_C"].sum())
efficiency = N_matched / N_eligible

print("Multi-chamber pseudo-tracklet estimate:")
print(f"Eligible events = {N_eligible}")
print(f"Matched events  = {N_matched}")
print(f"Efficiency      = {efficiency:.4f}")



## 8. Residual distributions

Residuals measure how far the measured target hit is from the predicted one:
$$
\Delta x = x_C^{\mathrm{meas}} - x_C^{\mathrm{pred}}, \qquad
\Delta y = y_C^{\mathrm{meas}} - y_C^{\mathrm{pred}}, \qquad
\Delta t = t_C^{\mathrm{meas}} - t_C^{\mathrm{pred}}.
$$

These plots are very useful because they help choose sensible tolerances.


In [ ]:

matched_or_seen = study["eligible_AB_to_C"] & (study["seen_C"] == True)

dx = (study.loc[matched_or_seen, "x_C"] - study.loc[matched_or_seen, "xC_pred"]).dropna()
dy = (study.loc[matched_or_seen, "y_C"] - study.loc[matched_or_seen, "yC_pred"]).dropna()
dt = (study.loc[matched_or_seen, "t_C"] - study.loc[matched_or_seen, "tC_pred"]).dropna()

plt.figure()
plt.hist(dx, bins=50)
plt.axvline(tol_x, linestyle="--")
plt.axvline(-tol_x, linestyle="--")
plt.xlabel(r"$\Delta x$ [cm]")
plt.ylabel("Counts")
plt.title("Residual distribution in x")
plt.show()

plt.figure()
plt.hist(dy, bins=50)
plt.axvline(tol_y, linestyle="--")
plt.axvline(-tol_y, linestyle="--")
plt.xlabel(r"$\Delta y$ [cm]")
plt.ylabel("Counts")
plt.title("Residual distribution in y")
plt.show()

plt.figure()
plt.hist(dt, bins=50)
plt.axvline(tol_t, linestyle="--")
plt.axvline(-tol_t, linestyle="--")
plt.xlabel(r"$\Delta t$ [ns]")
plt.ylabel("Counts")
plt.title("Residual distribution in time")
plt.show()



## 9. Event-by-event example

Let us inspect a few events to see the prediction and the measurement side by side.


In [ ]:

cols = [
    "event",
    "seen_A", "x_A", "y_A", "t_A",
    "seen_B", "x_B", "y_B", "t_B",
    "seen_C", "x_C", "y_C", "t_C",
    "xC_pred", "yC_pred", "tC_pred",
    "matched_AB_to_C",
]

study.loc[study["eligible_AB_to_C"], cols].head(10)



## 10. A compact efficiency function

It is useful for students to have the full logic wrapped into a single function.


In [ ]:

def compute_pseudotracklet_efficiency(df, ref1, ref2, target, z_positions, tol_x, tol_y, tol_t):
    work = df.copy()

    eligible = (
        (work[f"seen_{ref1}"] == True) &
        (work[f"seen_{ref2}"] == True)
    )

    x_pred, y_pred, t_pred = predict_target_from_two_refs(work, ref1, ref2, target, z_positions)

    matched = (
        eligible &
        (work[f"seen_{target}"] == True) &
        (np.abs(work[f"x_{target}"] - x_pred) < tol_x) &
        (np.abs(work[f"y_{target}"] - y_pred) < tol_y) &
        (np.abs(work[f"t_{target}"] - t_pred) < tol_t)
    )

    n_eligible = int(eligible.sum())
    n_matched = int(matched.sum())
    eff = n_matched / n_eligible if n_eligible > 0 else np.nan

    return {
        "n_eligible": n_eligible,
        "n_matched": n_matched,
        "efficiency": eff,
    }

result = compute_pseudotracklet_efficiency(
    data, ref1="A", ref2="B", target="C",
    z_positions=z_positions, tol_x=tol_x, tol_y=tol_y, tol_t=tol_t
)

result



## 11. Simple HV scan model for students

In real RPC studies, efficiency is measured as a function of the high voltage (HV).

To keep the notebook independent from any specific repository, we simulate this in a simple way:
- the detector efficiency of chamber C increases with HV,
- we repeat the same pseudo-tracklet logic for each HV point.


In [ ]:

hv_points = np.array([6.0, 6.5, 6.7, 6.8, 6.9, 7.0, 7.1, 7.2, 7.3, 7.4, 7.5])

def logistic_efficiency(hv, hv50=6.85, k=8.0, plateau=0.96):
    return plateau / (1 + np.exp(-k * (hv - hv50)))

hv_results = []

for hv in hv_points:
    eff_C = float(logistic_efficiency(hv))
    effs_hv = efficiencies.copy()
    effs_hv["C"] = eff_C

    hits_hv = simulate_hits(truth, z_positions, effs_hv, sigma_x, sigma_y, sigma_t, rng)
    wide_hv = hits_hv.pivot(index="event", columns="chamber", values=["seen", "x", "y", "t"])
    wide_hv.columns = [f"{name}_{ch}" for name, ch in wide_hv.columns]
    wide_hv = wide_hv.reset_index()
    data_hv = truth.merge(wide_hv, on="event")

    out = compute_pseudotracklet_efficiency(
        data_hv, ref1="A", ref2="B", target="C",
        z_positions=z_positions, tol_x=tol_x, tol_y=tol_y, tol_t=tol_t
    )
    out["hv_kv"] = hv
    hv_results.append(out)

hv_df = pd.DataFrame(hv_results)
hv_df



## 12. Efficiency versus HV

This is the key summary plot in many detector studies.


In [ ]:

plt.figure()
plt.plot(hv_df["hv_kv"], hv_df["efficiency"], marker="o")
plt.xlabel("High Voltage [kV]")
plt.ylabel("Efficiency")
plt.title("Simplified pseudo-tracklet efficiency vs HV")
plt.ylim(0, 1.05)
plt.grid(alpha=0.3)
plt.show()



## 13. Saving the results

Students often benefit from saving outputs into a simple table.


In [ ]:

hv_df.to_csv("simplified_rpc_hv_scan.csv", index=False)
print("Saved: simplified_rpc_hv_scan.csv")



## 14. What students should retain from this notebook

### Physics message
The efficiency of one chamber can be estimated by using the others to define a candidate trajectory.

### Algorithmic message
A pseudo-tracklet method needs:
1. reference chambers,
2. a prediction in the target chamber,
3. compatibility cuts,
4. a count of eligible and matched events.

### Computational message
Even a detector problem can be broken into very simple steps:
- simulate or read data,
- reconstruct,
- compare prediction and measurement,
- count,
- plot.

## Natural extensions
A future version can include:
- real ROOT input,
- strip clustering,
- centroid reconstruction,
- 3-chamber or 4-chamber variants,
- 2D efficiency maps,
- binomial uncertainty bars.
